<a href="https://colab.research.google.com/github/Kaoutarash8/BERT_ARABERT_ASAHAAD_DAOUAD/blob/main/AraBERT_vs_mBERT_ASTD_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparaison mBERT vs AraBERT — ASTD
Classification de sentiment en arabe. 3 epoques, MAX_LENGTH=64, 50% du dataset.

In [1]:
!pip install transformers datasets torch scikit-learn accelerate -q

In [2]:
import torch, numpy as np, pandas as pd, transformers, warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

T_VERSION = tuple(int(x) for x in transformers.__version__.split('.')[:2])

SEED        = 42
LABEL_NAMES = ['Neutral', 'Objective', 'Positive', 'Negative']
NUM_LABELS  = 4
MAX_LENGTH  = 64
BATCH_SIZE  = 32
NUM_EPOCHS  = 3
SAMPLE_FRAC = 0.50

np.random.seed(SEED)
torch.manual_seed(SEED)
print('ok')

ok


## 1. Chargement du dataset

In [3]:
raw = load_dataset('arbml/ASTD')
df  = raw['train'].to_pandas()

# echantillonnage stratifie 50%
df, _ = train_test_split(df, train_size=SAMPLE_FRAC, random_state=SEED, stratify=df['label'])
df = df.reset_index(drop=True)
print(f'{len(df)} exemples apres echantillonnage')

# split 70 / 15 / 15
train_df, tmp = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['label'])
val_df, test_df = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=tmp['label'])
print(f'train={len(train_df)}  val={len(val_df)}  test={len(test_df)}')

def to_hf(d):
    return Dataset.from_dict({'tweet': d['tweet'].tolist(), 'label': d['label'].tolist()})

splits = DatasetDict({'train': to_hf(train_df), 'validation': to_hf(val_df), 'test': to_hf(test_df)})

README.md:   0%|          | 0.00/431 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/941k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9694 [00:00<?, ? examples/s]

4847 exemples apres echantillonnage
train=3392  val=727  test=728


## 2. Fonctions communes

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy'       : accuracy_score(labels, preds),
        'f1_macro'       : f1_score(labels, preds, average='macro'),
        'f1_weighted'    : f1_score(labels, preds, average='weighted'),
        'precision_macro': precision_score(labels, preds, average='macro', zero_division=0),
        'recall_macro'   : recall_score(labels, preds, average='macro', zero_division=0),
    }

def make_trainer(model, args, train_ds, val_ds, collator, tokenizer):
    kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
              data_collator=collator, compute_metrics=compute_metrics)
    kw['processing_class' if T_VERSION >= (4, 46) else 'tokenizer'] = tokenizer
    return Trainer(**kw)

def get_training_args(output_dir):
    eval_kw = 'eval_strategy' if T_VERSION >= (4, 41) else 'evaluation_strategy'
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=2e-5,
        weight_decay=0.01,
        **{eval_kw: 'epoch'},
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        logging_steps=100,
        report_to='none',
        seed=SEED,
        fp16=torch.cuda.is_available(),
    )

print('ok')

ok


## 3. Fine-tuning

In [5]:
MODELS = {
    'mBERT'  : 'bert-base-multilingual-cased',
    'AraBERT': 'aubmindlab/bert-base-arabertv2',
}

results = {}

for name, model_id in MODELS.items():
    print(f'\n--- {name} ({model_id}) ---')

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    def tokenize(batch):
        return tokenizer(batch['tweet'], truncation=True, max_length=MAX_LENGTH, padding=False)

    tok = splits.map(tokenize, batched=True, remove_columns=['tweet'])
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=NUM_LABELS,
        id2label={i: l for i, l in enumerate(LABEL_NAMES)},
        label2id={l: i for i, l in enumerate(LABEL_NAMES)},
    )

    trainer = make_trainer(model, get_training_args(f'./out_{name}'),
                           tok['train'], tok['validation'], collator, tokenizer)

    trainer.train()

    out    = trainer.predict(tok['test'])
    y_pred = np.argmax(out.predictions, axis=-1)
    y_true = out.label_ids

    results[name] = {
        'Accuracy'       : round(accuracy_score(y_true, y_pred), 4),
        'F1 Macro'       : round(f1_score(y_true, y_pred, average='macro'), 4),
        'F1 Weighted'    : round(f1_score(y_true, y_pred, average='weighted'), 4),
        'Precision Macro': round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'Recall Macro'   : round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
    }

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()


--- mBERT (bert-base-multilingual-cased) ---


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro
1,0.904676,0.811432,0.696011,0.338879,0.643551,0.305737,0.386740
2,0.758010,0.761191,0.719395,0.415655,0.678887,0.581752,0.424571
3,0.675487,0.751925,0.718019,0.454374,0.687886,0.531370,0.442523


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

## 4. Tableau comparatif

In [6]:
table = pd.DataFrame(results).T
print(table.to_string())

         Accuracy  F1 Macro  F1 Weighted  Precision Macro  Recall Macro
mBERT      0.6854    0.2971       0.6107           0.2900        0.3159
AraBERT    0.7321    0.4630       0.6989           0.5412        0.4423


# Remarque :
Les performances absolues sont sous-optimales car l'entraînement a été fait sur seulement 50% du dataset (contrainte de temps sur Google Colab). Cependant, la comparaison relative entre les deux modèles montrent:

 qu'AraBERT est plus performant que mBERT sur toutes les métriques. AraBERT atteint une précision (accuracy) de 73,2% contre 68,5% pour mBERT. L'écart est encore plus marqué sur le F1 Macro : 46,3% pour AraBERT contre seulement 29,7% pour mBERT.

Cette différence s'explique par la spécialisation : AraBERT est un modèle spécifiquement entraîné sur la langue arabe, tandis que mBERT est un modèle généraliste qui couvre plus de 100 langues. Pour une tâche en arabe, le modèle spécialisé comprend mieux les nuances linguistiques.
